# Credit Scoring Tool - Notebook de Ejemplo

Esta notebook demuestra todas las funcionalidades de la paquetería **credit-scoring-tool** usando el dataset `application_train.csv`.

**Secciones:**
1. Carga de Datos
2. Preprocesamiento (Missing Values, Encoding, Binning, WOE)
3. Selección de Variables (IV, Correlación)
4. Modelos (Logistic Regression, Random Forest, Neural Network, XGBoost)
5. Evaluación (Métricas de Clasificación y Credit Scoring)
6. Visualización
7. Pipeline Integrado

---
## 1. Carga de Datos

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Cargamos y preparamos datos
datos = pd.read_csv('application_train.csv')

X = datos.drop(columns=['TARGET', 'SK_ID_CURR'])
y = datos['TARGET']

print(f"Dataset shape: {X.shape}")
print(f"Target distribution:{y.value_counts(normalize=True)}")
X.head()

Dataset shape: (307511, 120)
Target distribution:TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,297000.0,Unaccompanied,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,513000.0,Unaccompanied,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [43]:
# División en train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (215257, 120), Test: (92254, 120)


---
## 2. Preprocesamiento

### 2.1 Manejo de Valores Faltantes

In [44]:
from creditScoring.preprocessing import (
    handle_missing_values,
    handle_numeric_missing_values,
    handle_categorical_missing_values,
)

# Ver valores faltantes antes
print("Valores faltantes por columna (top 10):" )
print(X_train.isnull().sum().sort_values(ascending=False).head(10))

Valores faltantes por columna (top 10):
COMMONAREA_MODE             150377
COMMONAREA_MEDI             150377
COMMONAREA_AVG              150377
NONLIVINGAPARTMENTS_MEDI    149407
NONLIVINGAPARTMENTS_AVG     149407
NONLIVINGAPARTMENTS_MODE    149407
FONDKAPREMONT_MODE          147183
LIVINGAPARTMENTS_AVG        147079
LIVINGAPARTMENTS_MODE       147079
LIVINGAPARTMENTS_MEDI       147079
dtype: int64


In [45]:
# Manejo de numéricos con mediana
X_num_clean = handle_numeric_missing_values(X_train, strategy="replace",value= -1)
print(f"Valores faltantes numéricos después de mediana: {X_num_clean.select_dtypes(include=['number']).isnull().sum().sum()}")

# Manejo de categóricos con moda
X_cat_clean = handle_categorical_missing_values(X_train, strategy="constant", fill_value="missing")
print(f"Valores faltantes categóricos después de constante: {X_cat_clean.select_dtypes(exclude=['number']).isnull().sum().sum()}")

Valores faltantes numéricos después de mediana: 0
Valores faltantes categóricos después de constante: 0


In [46]:
# Función combinada
X_train_clean = handle_missing_values(
    X_train,
    numeric_strategy="replace",
    numeric_fill_value=-1,
    categorical_strategy="constant",
    categorical_fill_value="missing",
)
X_test_clean = handle_missing_values(
    X_test,
    numeric_strategy="replace",
    numeric_fill_value=-1,
    categorical_strategy="constant",
    categorical_fill_value="missing",
)
print(f"Total missing después de limpieza (train): {X_train_clean.isnull().sum().sum()}")
print(f"Total missing después de limpieza (test): {X_test_clean.isnull().sum().sum()}")

Total missing después de limpieza (train): 0
Total missing después de limpieza (test): 0


### 2.2 Encoding Categórico

In [47]:
from creditScoring.preprocessing import CategoricalEncoder

# Encoding por frecuencia
encoder = CategoricalEncoder(method="onehot")
encoder.fit(X_train_clean, y_train)
X_train_encoded = encoder.transform(X_train_clean)
X_test_encoded = encoder.transform(X_test_clean)

print(f"Columnas categóricas codificadas: {len(encoder.columns_)}")
print(f"Shape después de encoding: {X_train_encoded.shape}")
X_train_encoded.head()

Columnas categóricas codificadas: 16
Shape después de encoding: (215257, 250)


,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,...,WALLSMATERIAL_MODE_Mixed,WALLSMATERIAL_MODE_Monolithic,WALLSMATERIAL_MODE_Others,WALLSMATERIAL_MODE_Panel,"WALLSMATERIAL_MODE_Stone, brick",WALLSMATERIAL_MODE_Wooden,WALLSMATERIAL_MODE_missing,EMERGENCYSTATE_MODE_No,EMERGENCYSTATE_MODE_Yes,EMERGENCYSTATE_MODE_missing
127044,0,157500.0,706410.0,67072.5,679500.0,0.032561,-14653,-2062,-8599.0,-2087,...,False,False,False,True,False,False,False,True,False,False
281143,1,121500.0,545040.0,25407.0,450000.0,0.007114,-13995,-2246,-348.0,-172,...,False,False,False,False,True,False,False,True,False,False
199799,1,225000.0,942300.0,27679.5,675000.0,0.022625,-21687,-1335,-6306.0,-4026,...,False,False,False,True,False,False,False,True,False,False
306749,2,144000.0,180000.0,9000.0,180000.0,0.006629,-13071,-2292,-742.0,-1201,...,False,False,False,False,False,False,False,True,False,False
301347,0,112500.0,729792.0,37390.5,630000.0,0.046220,-19666,365243,-169.0,-3112,...,False,False,False,False,False,False,True,False,False,True


### 2.3 Binning / Discretización

In [52]:
from creditScoring.preprocessing import BinningTransformer

# Binning monótono (requiere target)
binner = BinningTransformer(method="monotonic", n_bins=5, min_bin_pct=0.05)
binner.fit(X_train_encoded, y_train)
X_train_binned = binner.transform(X_train_encoded)

print(f"Columnas numéricas binneadas: {len(binner.numeric_columns_)}")
print(f"Ejemplo de bins para la primera columna numérica:")
col_ejemplo = binner.numeric_columns_[1]
print(f"  {col_ejemplo}: {X_train_binned[col_ejemplo].value_counts(normalize=True).sort_index()}")

Columnas numéricas binneadas: 104
Ejemplo de bins para la primera columna numérica:
  AMT_INCOME_TOTAL: AMT_INCOME_TOTAL
2    0.207473
3    0.278941
4    0.114798
1    0.244893
0    0.153895
Name: proportion, dtype: float64


In [53]:
binner._bin_mappings_

{'CNT_CHILDREN': {Interval(-inf, 1.0, closed='right'): 0,
  Interval(1.0, inf, closed='right'): 1},
 'AMT_INCOME_TOTAL': {Interval(225000.0, inf, closed='right'): 0,
  Interval(162000.0, 225000.0, closed='right'): 1,
  Interval(-inf, 99000.0, closed='right'): 2,
  Interval(99000.0, 135000.0, closed='right'): 3,
  Interval(135000.0, 162000.0, closed='right'): 4},
 'AMT_CREDIT': {Interval(900000.0, inf, closed='right'): 0,
  Interval(-inf, 254700.0, closed='right'): 1,
  Interval(607041.0, 900000.0, closed='right'): 2,
  Interval(254700.0, 431280.0, closed='right'): 3,
  Interval(431280.0, 607041.0, closed='right'): 4},
 'AMT_ANNUITY': {Interval(37575.0, inf, closed='right'): 0,
  Interval(-inf, 14683.5, closed='right'): 1,
  Interval(14683.5, 21865.5, closed='right'): 2,
  Interval(21865.5, 28062.0, closed='right'): 3,
  Interval(28062.0, 37575.0, closed='right'): 4},
 'AMT_GOODS_PRICE': {Interval(823500.0, inf, closed='right'): 0,
  Interval(522000.0, 823500.0, closed='right'): 1,
  In

### 2.4 WOE (Weight of Evidence)

In [58]:
from creditScoring.preprocessing import WOETransformer, calculate_woe_iv, calculate_iv_for_dataframe

# Numero de columna de EXT_SOURCE_3
col_ejemplo = X_train_str.columns.get_loc('EXT_SOURCE_3')

# Calcular WOE/IV para una variable individual (EXT_SOURCE_3)
X_train_str = X_train_binned.astype(str)
ejemplo_col = X_train_str.columns[col_ejemplo]
woe_table, iv_value = calculate_woe_iv(X_train_str[ejemplo_col], y_train)
print(f"WOE table para '{ejemplo_col}' (IV={iv_value:.4f}):")
print(woe_table[['bin', 'woe', 'iv_component']].to_string(index=False))

WOE table para 'EXT_SOURCE_3' (IV=0.2930):
bin       woe  iv_component
  0 -0.872211      0.106839
  1 -0.539152      0.046575
  2 -0.077285      0.001152
  3  0.163022      0.005663
  4  0.702273      0.132760


In [57]:
# Transformador WOE completo
woe_transformer = WOETransformer()
woe_transformer.fit(X_train_str, y_train)
X_train_woe = woe_transformer.transform(X_train_str)

print(f"Top 10 variables por IV:")
print(woe_transformer.iv_table_.head(10).to_string(index=False))

Top 10 variables por IV:
                             feature       iv
                        EXT_SOURCE_3 0.292989
                        EXT_SOURCE_2 0.275664
                       DAYS_EMPLOYED 0.097259
                        EXT_SOURCE_1 0.085416
                          DAYS_BIRTH 0.081793
NAME_EDUCATION_TYPE_Higher education 0.047676
                     AMT_GOODS_PRICE 0.045052
            NAME_INCOME_TYPE_Working 0.042742
              DAYS_LAST_PHONE_CHANGE 0.041406
                       CODE_GENDER_M 0.038774


---
## 3. Selección de Variables

### 3.1 Selección por Information Value (IV)

In [60]:
from creditScoring.feature_selection import IVFeatureSelector

# Selección basada en IV mínimo
iv_selector = IVFeatureSelector(min_iv=0.02)
iv_selector.fit(X_train_woe, y_train)

print(f"Variables seleccionadas (IV >= 0.02): {len(iv_selector.selected_features_)}")
print(f"Variables descartadas: {X_train_woe.shape[1] - len(iv_selector.selected_features_)}")
print(f"Ranking IV (top 15):")
print(iv_selector.get_ranking().head(10).to_string(index=False))

Variables seleccionadas (IV >= 0.02): 47
Variables descartadas: 203
Ranking IV (top 15):
                             feature       iv
                        EXT_SOURCE_3 0.292989
                        EXT_SOURCE_2 0.275664
                       DAYS_EMPLOYED 0.097259
                        EXT_SOURCE_1 0.085416
                          DAYS_BIRTH 0.081793
NAME_EDUCATION_TYPE_Higher education 0.047676
                     AMT_GOODS_PRICE 0.045052
            NAME_INCOME_TYPE_Working 0.042742
              DAYS_LAST_PHONE_CHANGE 0.041406
                       CODE_GENDER_M 0.038774


### 3.2 Eliminación por Correlación

In [61]:
from creditScoring.feature_selection import correlation_matrix, remove_correlated_features

# Matriz de correlación
X_selected = iv_selector.transform(X_train_woe)
corr = correlation_matrix(X_selected, method="pearson")
print(f"Shape matriz de correlación: {corr.shape}")

# Eliminar features altamente correlacionadas
ranking = iv_selector.get_ranking()
final_features = remove_correlated_features(
    X_selected, ranking, threshold=0.8, method="pearson"
)
print(f"Features después de filtro de correlación (threshold=0.8): {len(final_features)}")
print(f"Features eliminadas por correlación: {len(iv_selector.selected_features_) - len(final_features)}")

Shape matriz de correlación: (47, 47)
Features después de filtro de correlación (threshold=0.8): 23
Features eliminadas por correlación: 24


---
## 4. Modelos

Probamos los 4 algoritmos disponibles en la paquetería.

### 4.1 Regresión Logística

In [62]:
from creditScoring.models import LogisticRegressionModel

# Preparar datos finales para modelos
X_train_final = X_train_woe[final_features]
X_test_str = X_test_encoded.astype(str)
X_test_woe = woe_transformer.transform(X_test_str)
X_test_final = X_test_woe[final_features]

# Entrenar Logistic Regression
lr_model = LogisticRegressionModel(random_state=42)
lr_model.fit(X_train_final, y_train)

# Predicciones
y_pred_lr = lr_model.predict(X_test_final)
y_prob_lr = lr_model.predict_proba(X_test_final)

# Coeficientes
coefs = lr_model.get_coefficients()
print("Top 10 coeficientes (valor absoluto):")
print(coefs.abs().sort_values(ascending=False).head(10))

Top 10 coeficientes (valor absoluto):
EXT_SOURCE_3                            0.853645
EXT_SOURCE_2                            0.756515
AMT_ANNUITY                             0.667747
NAME_EDUCATION_TYPE_Higher education    0.654274
CODE_GENDER_M                           0.604187
EXT_SOURCE_1                            0.550666
DAYS_EMPLOYED                           0.447069
REGION_RATING_CLIENT_W_CITY             0.416615
AMT_GOODS_PRICE                         0.405372
DAYS_ID_PUBLISH                         0.325346
dtype: float64


### 4.2 Random Forest

In [63]:
from creditScoring.models import RandomForestModel

rf_model = RandomForestModel(n_estimators=100, random_state=42)
rf_model.fit(X_train_final, y_train)

y_pred_rf = rf_model.predict(X_test_final)
y_prob_rf = rf_model.predict_proba(X_test_final)

# Feature importance
importances = rf_model.feature_importance()
print("Top 10 features por importancia:")
print(importances.sort_values(ascending=False).head(10))

Top 10 features por importancia:
DAYS_REGISTRATION             0.090903
REGION_POPULATION_RELATIVE    0.089368
DAYS_LAST_PHONE_CHANGE        0.084218
DAYS_ID_PUBLISH               0.080684
AMT_ANNUITY                   0.075353
AMT_GOODS_PRICE               0.072385
EXT_SOURCE_3                  0.064055
EXT_SOURCE_2                  0.062499
DAYS_BIRTH                    0.059047
DAYS_EMPLOYED                 0.053899
dtype: float64


### 4.3 Red Neuronal (MLP)

In [64]:
from creditScoring.models import NeuralNetworkModel

nn_model = NeuralNetworkModel(
    hidden_layer_sizes=(64, 32),
    max_iter=300,
    random_state=42,
)
nn_model.fit(X_train_final, y_train)

y_pred_nn = nn_model.predict(X_test_final)
y_prob_nn = nn_model.predict_proba(X_test_final)

print(f"Predicciones generadas: {len(y_pred_nn)}")
print(f"Rango de probabilidades: [{y_prob_nn.min():.4f}, {y_prob_nn.max():.4f}]")

Predicciones generadas: 92254
Rango de probabilidades: [0.0228, 0.2999]


### 4.4 XGBoost

In [65]:
from creditScoring.models import XGBoostModel

xgb_model = XGBoostModel(
    n_estimators=100,
    learning_rate=0.02,
    max_depth=5,
    random_state=42,
)
xgb_model.fit(X_train_final, y_train)

y_pred_xgb = xgb_model.predict(X_test_final)
y_prob_xgb = xgb_model.predict_proba(X_test_final)

print(f"Predicciones generadas: {len(y_pred_xgb)}")
print(f"Rango de probabilidades: [{y_prob_xgb.min():.4f}, {y_prob_xgb.max():.4f}]")

Predicciones generadas: 92254
Rango de probabilidades: [0.0405, 0.0637]


---
## 5. Evaluación

### 5.1 ROC modelos

In [78]:
# Roc de los modelos impreso
from sklearn.metrics import roc_auc_score
print("Roc de los modelos:")
print("Logistic Regression:", roc_auc_score(y_test, y_prob_lr))
print("Random Forest:", roc_auc_score(y_test, y_prob_rf))
print("Neural Network:", roc_auc_score(y_test, y_prob_nn))
print("XGBoost:", roc_auc_score(y_test, y_prob_xgb))

Roc de los modelos:
Logistic Regression: 0.5831748604808379
Random Forest: 0.5442710529660713
Neural Network: 0.5875000419166738
XGBoost: 0.5808488845703969


---
## 7. Pipeline Integrado

La clase  integra todo el flujo de preprocesamiento, selección de variables, entrenamiento y evaluación.

### 7.1 Pipeline con Regresión Logística (default)

In [ ]:
from creditScoring import CreditScoringPipeline, PipelineConfig, get_default_config

# Configuración por defecto (logistic regression)
config = get_default_config()
print("Configuración por defecto:")
for key, value in config.items():
    print(f"  {key}: {value}")

In [ ]:
# Crear y entrenar pipeline
pipeline_lr = CreditScoringPipeline(config)
pipeline_lr.fit(X_train, y_train)

# Evaluar
results_lr = pipeline_lr.evaluate(X_test, y_test)
print("Pipeline Logistic Regression:")
print(results_lr.summary())

### 7.2 Pipeline con Random Forest

In [ ]:
config_rf = PipelineConfig(model_type="random_forest")
pipeline_rf = CreditScoringPipeline(config_rf)
pipeline_rf.fit(X_train, y_train)

results_rf = pipeline_rf.evaluate(X_test, y_test)
print("Pipeline Random Forest:")
print(results_rf.summary())

### 7.3 Pipeline con Neural Network

In [ ]:
config_nn = PipelineConfig(model_type="neural_network")
pipeline_nn = CreditScoringPipeline(config_nn)
pipeline_nn.fit(X_train, y_train)

results_nn = pipeline_nn.evaluate(X_test, y_test)
print("Pipeline Neural Network:")
print(results_nn.summary())

### 7.4 Pipeline con XGBoost

In [ ]:
config_xgb = PipelineConfig(model_type="xgboost")
pipeline_xgb = CreditScoringPipeline(config_xgb)
pipeline_xgb.fit(X_train, y_train)

results_xgb = pipeline_xgb.evaluate(X_test, y_test)
print("Pipeline XGBoost:")
print(results_xgb.summary())

### 7.5 Comparación Final de Pipelines

In [ ]:
comparacion = pd.DataFrame([
    {"Pipeline": "Logistic Regression", **results_lr.credit_metrics},
    {"Pipeline": "Random Forest", **results_rf.credit_metrics},
    {"Pipeline": "Neural Network", **results_nn.credit_metrics},
    {"Pipeline": "XGBoost", **results_xgb.credit_metrics},
])
print("Comparación de Pipelines (Métricas Credit Scoring):")
print(comparacion.to_string(index=False))

### 7.6 Guardar y Cargar Pipeline

In [ ]:
# Guardar el mejor pipeline
pipeline_xgb.save("pipeline_xgboost.joblib")
print("Pipeline guardado en 'pipeline_xgboost.joblib'")

# Cargar pipeline
pipeline_loaded = CreditScoringPipeline.load("pipeline_xgboost.joblib")
results_loaded = pipeline_loaded.evaluate(X_test, y_test)
print("Pipeline cargado - verificación:")
print(results_loaded.summary())

---
## Resumen

Esta notebook demostró las siguientes funcionalidades de **credit-scoring-tool**:

| Módulo | Funcionalidades |
|--------|----------------|
|  | Missing values, Encoding categórico, Binning, WOE/IV |
|  | Selección por IV, Filtro de correlación |
|  | Logistic Regression, Random Forest, Neural Network, XGBoost |
|  | Clasificación (accuracy, precision, recall, F1, AUC), Credit Scoring (KS, Gini, PSI, Lift, Divergence) |
|  | Distribuciones, Correlación, Scores, ROC, KS, Feature Importance, Lift/Gain |
|  | Pipeline integrado end-to-end con save/load |